# 23. Hugging Face 기초 — 사전학습 모델 다루기

> **제23장** · **이론편 대응: 19장(Foundation Model), 20.1~20.3절(LLM 아키텍처·토큰화)**
> **예상 소요**: 70분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: **transformers** (1절 참조)
> **다운로드**: GPT-2 약 550MB (자동)

---

## 이 장에서 하는 일

22장까지 Transformer를 밑바닥부터 만들었다. 이제 **실제로 학습된 모델**을 가져다 쓴다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | **transformers 설치** | — |
| 1 | 모델 내려받기 | 19.1절 |
| 2 | **토크나이저 — BPE 확인** ★ | 20.2절 |
| 3 | 언어별 토큰 효율 | 20.2절 |
| 4 | **모델 내부 뜯어보기 — 우리가 만든 것과 대조** ★ | 18장 |
| 5 | 추론 실행 | 20.1절 |
| 6 | Attention 가중치 확인 | 18.3절 |
| 7 | 모델 크기와 메모리 | 22.5절 |

**4절이 이 장의 핵심이다.** 21~22장에서 만든 부품이
실제 모델 안에 그대로 들어 있다는 것을 확인한다.

---

## 1. 준비 — transformers 설치

### Hugging Face란

**모델과 데이터셋을 공유하는 플랫폼**이자, 그것을 다루는 라이브러리다.
수십만 개의 사전학습 모델이 올라와 있고, 몇 줄이면 가져다 쓸 수 있다.

| 항목 | 내용 |
|---|---|
| 라이브러리 | `transformers` |
| 모델 저장소 | `https://huggingface.co/models` |
| 문서 | `https://huggingface.co/docs/transformers` |

### 설치

터미널에서 (가상환경 활성화 상태로) 실행한다.

```
pip install transformers
```

함께 설치되는 것들:
- `tokenizers` — 빠른 토큰화 (Rust로 구현)
- `huggingface-hub` — 모델 내려받기
- `safetensors` — 모델 가중치 형식

### 모델은 어디에 저장되나

처음 모델을 부르면 자동으로 내려받아 **캐시 폴더**에 저장한다.

| OS | 기본 위치 |
|---|---|
| Windows | `C:\Users\사용자\.cache\huggingface\hub` |
| Linux/Mac | `~/.cache/huggingface/hub` |

**한 번 받으면 다시 받지 않는다.** 여러 장에서 같은 모델을 써도 한 번만 내려받는다.

위치를 바꾸고 싶다면 환경 변수 `HF_HOME`을 설정한다.

### 이 장에서 쓸 모델

**GPT-2 (약 550MB)** — 2019년 공개된 모델로, 지금 기준으로는 작지만
**구조가 오늘날 LLM과 같아** 학습용으로 적합하다. CPU에서도 돌아간다.

In [ ]:
import importlib
import sys

print("=" * 60)
print("필요 패키지 확인")
print("=" * 60)

required = [
    ("transformers", "Hugging Face 라이브러리", "pip install transformers"),
    ("torch",        "PyTorch", "pip install torch --index-url https://download.pytorch.org/whl/cu128"),
    ("tokenizers",   "빠른 토큰화 (자동 설치)", "transformers와 함께"),
]

missing = []
for name, desc, install in required:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "?")
        print(f"[OK]   {name:<16}{ver:<12}{desc}")
    except ImportError:
        print(f"[없음] {name:<16}{'':<12}{desc}")
        missing.append(install)

print("-" * 60)
if missing:
    print("설치가 필요합니다:")
    for cmd in set(missing):
        print(f"  {cmd}")
    print()
    print("설치 후 커널을 재시작하세요.")
else:
    print("[준비 완료] 2절로 진행하세요.")
    print()
    # 캐시 위치 확인
    from pathlib import Path
    import os
    cache = os.environ.get("HF_HOME") or (Path.home() / ".cache" / "huggingface")
    print(f"모델 캐시 위치: {cache}")
    if Path(cache).exists():
        size = sum(f.stat().st_size for f in Path(cache).rglob("*") if f.is_file())
        print(f"현재 캐시 크기: {size/1024**2:.0f} MB")
    else:
        print("(아직 비어 있음 — 2절에서 모델을 받으면 생성됩니다)")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

import transformers
print(f"transformers {transformers.__version__}")

---

## 2. 모델 내려받기 — 이론편 19.1절

`AutoTokenizer`와 `AutoModel` 계열을 쓰면 **모델 이름만으로** 알아서 맞는 클래스를 골라 준다.

```python
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
```

`AutoModelFor___` 뒤에 붙는 것이 **용도**다.

| 클래스 | 용도 |
|---|---|
| `AutoModelForCausalLM` | 다음 토큰 예측 (GPT 계열) |
| `AutoModelForSequenceClassification` | 문장 분류 |
| `AutoModelForQuestionAnswering` | 질의응답 |
| `AutoModel` | 마지막 층 없이 표현만 |

같은 모델이라도 용도에 따라 마지막 층이 달라진다. 이것이 이론편 19.1절에서 다룬
**사전학습-미세조정 패러다임**의 구현이다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import time

MODEL_NAME = "gpt2"

print("=" * 60)
print(f"{MODEL_NAME} 내려받기")
print("=" * 60)
print("처음 실행하면 약 550MB를 내려받습니다 (몇 분 걸릴 수 있음)")
print("두 번째부터는 캐시에서 즉시 불러옵니다.")
print()

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"토크나이저 로드: {time.time()-t0:.1f}초")

t0 = time.time()
# ── from_pretrained() 파라미터 ───────────────────────────────
#   pretrained_model_name_or_path  모델 이름 또는 경로
#                                  예: 'gpt2', './my_model'
#   torch_dtype      가중치 자료형.  기본값 None(FP32)
#                    torch.float16 / torch.bfloat16 → 메모리 절반
#   device_map       장치 배치.  기본값 None
#                    'auto' 로 하면 GPU/CPU에 자동 분산
#   low_cpu_mem_usage  로딩 시 메모리 절약.  기본값 False
#   quantization_config  양자화 설정 (BitsAndBytesConfig)
#   attn_implementation  Attention 구현.  기본값 자동
#                        'eager'(내부 관찰 가능) / 'sdpa'(빠름)
#   cache_dir        캐시 위치.  기본값 ~/.cache/huggingface
#   trust_remote_code  커스텀 코드 실행 허용.  기본값 False
#                      **신뢰할 수 있는 모델에만 True**
# ──────────────────────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()
print(f"모델 로드      : {time.time()-t0:.1f}초")
print()

config = model.config
print("모델 정보")
print(f"  파라미터 수 : {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print(f"  층 수       : {config.n_layer}")
print(f"  Attention 헤드: {config.n_head}")
print(f"  d_model     : {config.n_embd}")
print(f"  어휘 크기   : {config.vocab_size:,}")
print(f"  최대 문맥   : {config.n_positions:,} 토큰")
print()
print("21~22장에서 만든 것과 같은 구조다.")
print(f"  우리 모델: d_model=64, 헤드 4, 층 2")
print(f"  GPT-2   : d_model={config.n_embd}, 헤드 {config.n_head}, 층 {config.n_layer}")

---

## 3. 토크나이저 — 이론편 20.2절 BPE 확인 ★

이론편 20.2절에서 BPE의 병합 과정을 손으로 계산했다. 말뭉치는 이랬다.

| 단어 | 빈도 |
|---|---|
| low | 5 |
| lower | 2 |
| newest | 6 |
| widest | 3 |

`(e, s)`가 9번으로 가장 빈번해 먼저 합쳐지고, 다음 라운드에 `est`가 된다고 했다.

**실제 GPT-2 토크나이저가 이 단어들을 어떻게 쪼개는지 확인해 보자.**

In [ ]:
print("=" * 60)
print("이론편 20.2절 예제 단어를 실제 토크나이저로")
print("=" * 60)
print(f"{'단어':<12}{'토큰 수':<10}{'분해 결과'}")
print("-" * 60)

for word in ["low", "lower", "newest", "widest"]:
    ids = tokenizer.encode(word)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"{word:<12}{len(ids):<10}{pieces}")

print("-" * 60)
print()
print("주목할 점")
print("  'newest' → ['new', 'est']")
print("  'widest' → ['wid', 'est']")
print()
print("두 단어 모두 'est'가 하나의 토큰으로 떨어졌다.")
print("이론편 20.2절에서 손으로 계산했던 그대로다 —")
print("자주 붙어 다니는 조각이 하나의 단위가 된 것이다.")
print()
print("문법 지식을 넣어 준 것이 아니라, 빈도만으로 찾아낸 결과다.")

In [ ]:
print("=" * 70)
print("하위단어 분해 — 처음 보는 단어도 다룰 수 있다")
print("=" * 70)

words = [
    "hello",
    "unbelievable",
    "tokenization",
    "antidisestablishmentarianism",
    "Transformer",
]

print(f"{'단어':<34}{'토큰 수':<10}{'분해'}")
print("-" * 70)
for w in words:
    ids = tokenizer.encode(w)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"{w:<34}{len(ids):<10}{pieces}")

print("-" * 70)
print()
print("긴 단어일수록 여러 조각으로 나뉜다.")
print("이 덕분에 사전에 없는 단어(신조어·오타)도 처리할 수 있다.")
print("→ 이론편 20.2절에서 다룬 '어휘 밖 단어(OOV) 문제'의 해결")

### 공백도 토큰의 일부다

GPT-2 토크나이저의 특징 중 하나다. **단어 앞의 공백이 토큰에 포함된다.**

In [ ]:
print("=" * 60)
print("공백 처리")
print("=" * 60)

pairs = [("hello", " hello"), ("world", " world")]

print(f"{'입력':<14}{'토큰 ID':<16}{'디코딩 결과'}")
print("-" * 60)
for a, b in pairs:
    for text in [a, b]:
        ids = tokenizer.encode(text)
        decoded = repr(tokenizer.decode(ids))
        print(f"{repr(text):<14}{str(ids):<16}{decoded}")
print("-" * 60)
print()
print("'hello'와 ' hello'가 서로 다른 토큰이다.")
print()
print("왜 이렇게 하는가")
print("  단어 경계 정보를 유지하면서도 토큰 수를 늘리지 않기 위해서다.")
print("  공백을 따로 토큰으로 만들면 문장 길이가 두 배 가까이 늘어난다.")
print()

# 문장 전체
sentence = "Hello, how are you today?"
ids = tokenizer.encode(sentence)
print(f"문장: {sentence}")
print(f"토큰 수: {len(ids)}")
print()
print(f"{'ID':<10}{'토큰'}")
print("-" * 30)
for i in ids:
    print(f"{i:<10}{repr(tokenizer.decode([i]))}")

---

## 4. 언어별 토큰 효율 — 이론편 20.2절

**같은 내용이라도 언어에 따라 토큰 수가 크게 다르다.**

GPT-2는 주로 영어 데이터로 학습되었으므로, 영어는 효율적으로 쪼개지고
다른 언어는 잘게 부서진다. 이것이 실무에서 비용 차이로 이어진다.

In [ ]:
print("=" * 70)
print("언어별 토큰 효율")
print("=" * 70)

texts = {
    "영어":   "Artificial intelligence is transforming the world.",
    "한국어": "인공지능이 세상을 바꾸고 있다.",
    "숫자":   "1234567890",
    "코드":   "def hello(): return 'world'",
}

print(f"{'언어':<10}{'글자 수':<10}{'토큰 수':<10}{'글자당 토큰':<14}{'효율'}")
print("-" * 70)
results = {}
for name, text in texts.items():
    ids = tokenizer.encode(text)
    ratio = len(ids) / len(text)
    results[name] = ratio
    bar = "█" * int(ratio * 10)
    print(f"{name:<10}{len(text):<10}{len(ids):<10}{ratio:<14.2f}{bar}")

print("-" * 70)
print()
print(f"한국어가 영어보다 글자당 {results['한국어']/results['영어']:.1f}배 많은 토큰을 쓴다.")
print()
print("이것이 뜻하는 것")
print("  1) 같은 내용이라도 한국어는 문맥 창을 더 많이 차지한다 (이론편 20.4절)")
print("  2) API 요금이 토큰 단위라면 비용이 더 든다")
print("  3) 생성 속도도 느려진다 (토큰마다 모델을 부르므로)")

In [ ]:
print("=" * 60)
print("한국어가 왜 잘게 쪼개지는가")
print("=" * 60)

korean = "안녕하세요"
ids = tokenizer.encode(korean)

print(f"'{korean}' → {len(ids)}개 토큰")
print()
print("각 토큰을 보면")
for i, tid in enumerate(ids[:8]):
    raw = tokenizer.decode([tid])
    print(f"  {i}: ID {tid:<8} {repr(raw)}")
if len(ids) > 8:
    print(f"  ... (총 {len(ids)}개)")
print()
print("글자가 아니라 바이트 단위로 쪼개지고 있다.")
print()
print("이유: GPT-2의 어휘에 한글 조각이 거의 없다.")
print("  학습 데이터가 대부분 영어였기 때문이다.")
print("  어휘에 없는 문자는 UTF-8 바이트 단위로 분해된다.")
print()
print("한국어를 많이 다룬다면")
print("  - 다국어 모델을 쓰거나")
print("  - 한국어 토크나이저를 별도로 학습시킨다")
print("  최근 모델들은 다국어 데이터를 포함해 이 문제가 많이 개선되었다.")

---

## 5. 모델 내부 뜯어보기 ★

**21~22장에서 만든 부품이 실제 모델 안에 그대로 있는지 확인한다.**

우리가 만든 구조를 떠올려 보자.

```
DecoderOnlyBlock:
  norm1 → MaskedSelfAttention → (+residual)
  norm2 → FeedForward → (+residual)
```

GPT-2의 블록도 같은지 보자.

In [ ]:
print("=" * 60)
print("모델 전체 구조")
print("=" * 60)

for name, module in model.named_children():
    print(f"  {name}: {type(module).__name__}")
print()

# 내부 접근 (버전에 따라 경로가 다를 수 있어 방어적으로)
transformer = getattr(model, "transformer", None) or getattr(model, "model", None)

print("Transformer 내부")
for name, module in transformer.named_children():
    if name == "h" or name == "layers":
        print(f"  {name}: {len(module)}개 블록")
    else:
        print(f"  {name}: {type(module).__name__}")
print()

blocks = getattr(transformer, "h", None) or getattr(transformer, "layers", None)
block0 = blocks[0]

print("=" * 60)
print("블록 하나의 구조")
print("=" * 60)
for name, module in block0.named_children():
    print(f"  {name:<10}{type(module).__name__}")
print()
print("우리가 만든 것과 대조 (22장)")
print(f"  {'GPT-2':<16}{'우리 구현':<24}{'역할'}")
print("-" * 60)
print(f"  {'ln_1':<16}{'norm1 (LayerNorm)':<24}{'정규화'}")
print(f"  {'attn':<16}{'MaskedSelfAttention':<24}{'마스크된 Self-Attention'}")
print(f"  {'ln_2':<16}{'norm2 (LayerNorm)':<24}{'정규화'}")
print(f"  {'mlp':<16}{'FeedForward':<24}{'위치별 변환'}")
print("-" * 60)
print()
print("[확인] 구조가 정확히 같다.")
print("  이름만 다를 뿐, 22장에서 만든 DecoderOnlyBlock 그대로다.")

In [ ]:
print("=" * 60)
print("Attention 내부")
print("=" * 60)

attn = block0.attn
for name, module in attn.named_children():
    shape = ""
    if hasattr(module, "weight"):
        shape = f"  가중치 {tuple(module.weight.shape)}"
    print(f"  {name:<12}{type(module).__name__}{shape}")
print()

print("Q, K, V 가중치가 하나로 합쳐져 있다")
if hasattr(attn, "c_attn"):
    w = attn.c_attn.weight
    print(f"  c_attn 가중치: {tuple(w.shape)}")
    print(f"  → {w.shape[0]} x {w.shape[1]} = d_model x (3 x d_model)")
    print(f"     {config.n_embd} x {3*config.n_embd}")
    print()
    print("우리는 W_q, W_k, W_v 를 따로 만들었지만,")
    print("실제 구현은 하나로 합쳐 한 번에 계산한다. 계산이 더 빠르기 때문이다.")
print()

print("=" * 60)
print("FFN 내부")
print("=" * 60)
mlp = block0.mlp
for name, module in mlp.named_children():
    shape = ""
    if hasattr(module, "weight"):
        shape = f"  {tuple(module.weight.shape)}"
    print(f"  {name:<12}{type(module).__name__}{shape}")
print()
print(f"d_model {config.n_embd} → 중간 {config.n_embd*4} → d_model {config.n_embd}")
print("21장에서 다룬 '4배로 늘렸다 줄이기'가 그대로 쓰인다.")

In [ ]:
import torch

print("=" * 60)
print("파라미터 분포")
print("=" * 60)

total = sum(p.numel() for p in model.parameters())

groups = {}
for name, p in model.named_parameters():
    if "wte" in name or "embed_tokens" in name:
        key = "토큰 임베딩"
    elif "wpe" in name or "embed_positions" in name:
        key = "위치 임베딩"
    elif "attn" in name:
        key = "Attention"
    elif "mlp" in name:
        key = "Feed-Forward"
    elif "ln" in name or "norm" in name:
        key = "LayerNorm"
    else:
        key = "기타"
    groups[key] = groups.get(key, 0) + p.numel()

print(f"{'구성 요소':<18}{'파라미터':<16}{'비율'}")
print("-" * 60)
for k, v in sorted(groups.items(), key=lambda x: -x[1]):
    bar = "█" * int(v / total * 40)
    print(f"{k:<18}{v:<16,}{v/total*100:5.1f}%  {bar}")
print("-" * 60)
print(f"{'합계':<18}{total:<16,}")
print()
print("Feed-Forward가 Attention보다 파라미터가 많다.")
print("  Attention: d_model x d_model x 4개")
print("  FFN      : d_model x 4d_model x 2개")
print()
print("Attention이 주목받지만, 파라미터의 절반 이상은 FFN에 있다.")

---

## 6. 추론 실행 — 이론편 20.1절

이제 실제로 문장을 생성해 본다. 이론편 20.1절에서 다룬 **자기회귀적 생성**이다.

22장 6절에서 직접 만든 `generate` 함수와 같은 일을 한다.

In [ ]:
import torch

print("=" * 60)
print("문장 생성")
print("=" * 60)

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")

print(f"입력: '{prompt}'")
print(f"토큰: {inputs['input_ids'][0].tolist()}")
print(f"      {[tokenizer.decode([i]) for i in inputs['input_ids'][0]]}")
print()

with torch.no_grad():
    # ── model.generate() 파라미터 ────────────────────────────────
    #   max_new_tokens  생성할 최대 토큰 수.  예: 50, 200, 500
    #                   max_length 와 달리 입력 길이를 뺀 값
    #   do_sample       샘플링 여부.  기본값 False(그리디)
    #                   False: 항상 최고 확률 토큰 → 결정적
    #                   True : 확률에 따라 뽑음 → 다양함
    #   temperature     확률 분포 조절.  기본값 1.0
    #                   낮을수록 안정(0.2~0.7), 높을수록 다양(1.0~1.5)
    #                   do_sample=True 일 때만 의미가 있다
    #   top_k           상위 k개만 후보.  기본값 50
    #                   예: 10(보수적) / 50(기본) / 0(비활성)
    #   top_p           누적 확률 p까지 후보.  기본값 1.0
    #                   예: 0.9, 0.95 — nucleus sampling
    #   repetition_penalty  반복 억제.  기본값 1.0
    #                       예: 1.1~1.2 — 같은 말 반복을 줄인다
    #   num_beams       빔 서치 폭.  기본값 1(비활성)
    #                   예: 4, 5 — 번역·요약에 유용, 느림
    #   pad_token_id    패딩 토큰 ID.  GPT-2 는 없으므로 eos 를 지정
    #   eos_token_id    종료 토큰 ID
    #   num_return_sequences  생성할 응답 개수.  기본값 1
    # ──────────────────────────────────────────────────────────────
    output = model.generate(
        **inputs,
        max_new_tokens=15,
        do_sample=False,          # 항상 가장 확률 높은 것 (탐욕적 선택)
        pad_token_id=tokenizer.eos_token_id,
    )

print("생성 결과")
print(f"  {tokenizer.decode(output[0])}")
print()
print(f"생성된 토큰 수: {output.shape[1] - inputs['input_ids'].shape[1]}")

In [ ]:
import torch

print("=" * 60)
print("한 단계씩 — 다음 토큰의 확률 보기")
print("=" * 60)

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits          # (배치, 토큰, 어휘)
print(f"출력 모양: {tuple(logits.shape)}")
print(f"  → 각 위치에서 어휘 {logits.shape[-1]:,}개에 대한 점수")
print()

# 마지막 위치의 예측만 사용
last_logits = logits[0, -1]
probs = torch.softmax(last_logits, dim=-1)

top_k = 10
values, indices = probs.topk(top_k)

print(f"'{prompt}' 다음에 올 토큰 상위 {top_k}개")
print(f"{'순위':<6}{'토큰':<20}{'확률':<12}{'막대'}")
print("-" * 60)
for rank, (v, idx) in enumerate(zip(values, indices), 1):
    token = repr(tokenizer.decode([idx]))
    bar = "█" * int(v.item() * 100)
    print(f"{rank:<6}{token:<20}{v.item():<12.4f}{bar}")
print("-" * 60)
print()
print(f"상위 10개의 확률 합: {values.sum().item():.4f}")
print(f"나머지 {logits.shape[-1]-top_k:,}개가 나머지 확률을 나눠 갖는다.")
print()
print("이것이 이론편 20.1절에서 다룬 '다음 토큰 예측'의 실제 모습이다.")

---

## 7. Attention 가중치 확인 — 이론편 18.3절

**20장에서 손으로 계산한 그 값**을 실제 모델에서 꺼내 본다.

`output_attentions=True`로 부르면 모든 층·모든 헤드의 가중치를 얻을 수 있다.

In [ ]:
import torch

text = "The cat sat on the mat"
inputs = tokenizer(text, return_tensors="pt")
tokens = [tokenizer.decode([i]) for i in inputs["input_ids"][0]]

# Attention 가중치를 보려면 attn_implementation="eager" 로 불러야 한다.
# 최신 transformers 는 기본이 sdpa(최적화 구현)인데,
# 이 방식은 속도를 위해 가중치 행렬을 만들지 않고 계산한다.
from transformers import AutoModelForCausalLM
model_eager = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, attn_implementation="eager")
model_eager.eval()

with torch.no_grad():
    outputs = model_eager(**inputs, output_attentions=True)

attentions = outputs.attentions      # 층별 튜플

if attentions is None or len(attentions) == 0:
    print("[주의] Attention 가중치를 얻지 못했습니다.")
    print("  attn_implementation='eager' 로 모델을 다시 불러오세요.")

print("=" * 60)
print("Attention 가중치 구조")
print("=" * 60)
print(f"층 개수    : {len(attentions)}")
print(f"각 층 모양 : {tuple(attentions[0].shape)}")
print(f"             (배치, 헤드, 토큰, 토큰)")
print()
print(f"토큰: {tokens}")
print()

# 20장에서 확인한 성질 — 각 행의 합이 1
first_layer = attentions[0][0]        # (헤드, 토큰, 토큰)
row_sums = first_layer.sum(dim=-1)
print(f"각 행의 합: 최소 {row_sums.min():.6f}, 최대 {row_sums.max():.6f}")
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-4)
print("[OK] 모든 행의 합이 1 — 20장에서 확인한 소프트맥스의 성질")
print()

# 마스킹 확인 (22장에서 만든 것)
print("미래를 보고 있는가 (22장 1절의 마스킹)")
upper = torch.triu(first_layer[0], diagonal=1)
print(f"  대각선 위쪽 최댓값: {upper.max().item():.10f}")
assert upper.max().item() < 1e-9
print("[OK] 미래 위치가 정확히 0 — GPT는 Decoder-only 구조")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# 여러 층·헤드의 패턴 비교
fig, axes = plt.subplots(2, 4, figsize=(15, 7))

picks = [(0, 0), (0, 5), (0, 11), (5, 0),
         (5, 5), (11, 0), (11, 5), (11, 11)]

for ax, (layer, head) in zip(axes.flat, picks):
    a = attentions[layer][0, head].numpy()
    im = ax.imshow(a, cmap="Blues", vmin=0, vmax=0.8)
    ax.set_title(f"층 {layer}, 헤드 {head}", fontsize=9)
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels([t.strip() for t in tokens], rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels([t.strip() for t in tokens], fontsize=7)

fig.suptitle("층·헤드마다 다른 Attention 패턴", fontsize=13)
plt.tight_layout()
plt.show()

print("모든 그림의 오른쪽 위가 비어 있다 = 마스킹")
print()
print("헤드마다 보는 방식이 다르다:")
print("  어떤 헤드는 바로 앞 토큰만 (대각선 바로 아래)")
print("  어떤 헤드는 첫 토큰에 몰림 (첫 열이 진함)")
print("  어떤 헤드는 넓게 퍼짐")
print()
print("21장에서 'Multi-Head가 여러 관계를 본다'고 한 것이 실제로 확인된다.")

---

## 8. 모델 크기와 메모리 — 이론편 22.5절

이론편 22.5절에서 모델 메모리를 계산했다. 실제 모델로 확인한다.

$$M = N_{params} \times \frac{b}{8}\text{ bytes}$$

In [ ]:
import torch

print("=" * 65)
print("메모리 계산 (이론편 22.5절)")
print("=" * 65)

n_params = sum(p.numel() for p in model.parameters())
actual_bytes = sum(p.numel() * p.element_size() for p in model.parameters())

print(f"파라미터 수 : {n_params:,}")
print(f"자료형      : {next(model.parameters()).dtype}")
print(f"실제 메모리 : {actual_bytes/1024**2:.1f} MB")
print()

print("정밀도별 예상 크기")
print(f"{'정밀도':<12}{'바이트/파라미터':<20}{'전체 크기'}")
print("-" * 65)
for name, bytes_per in [("FP32", 4), ("FP16", 2), ("INT8", 1), ("INT4", 0.5)]:
    size = n_params * bytes_per / 1024**2
    print(f"{name:<12}{bytes_per:<20}{size:>8.1f} MB")
print("-" * 65)
print()

print("더 큰 모델이라면 (이론편 22.5절 표)")
print(f"{'모델':<10}{'FP16':<14}{'INT4':<14}{'8GB GPU'}")
print("-" * 65)
for n, name in [(0.124e9, "GPT-2"), (1.5e9, "1.5B"), (7e9, "7B"), (13e9, "13B")]:
    fp16 = n * 2 / 1024**3
    int4 = n * 0.5 / 1024**3
    ok = "가능" if fp16 < 6.5 else ("INT4로 가능" if int4 < 6.5 else "어려움")
    print(f"{name:<10}{fp16:>7.1f} GB    {int4:>7.1f} GB    {ok}")
print("-" * 65)
print()
print("GPT-2는 작아서 CPU로도 충분하다.")
print("7B 이상을 다루려면 양자화가 필요하다 (이론편 22.5절).")

In [ ]:
import torch
import time

print("=" * 60)
print("추론 속도 측정")
print("=" * 60)

prompt = "Machine learning is"
inputs = tokenizer(prompt, return_tensors="pt")

for n_tokens in [10, 20, 40]:
    t0 = time.time()
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=n_tokens,
                           do_sample=False, pad_token_id=tokenizer.eos_token_id)
    elapsed = time.time() - t0
    print(f"  {n_tokens:3}토큰 생성: {elapsed:6.2f}초  "
          f"({n_tokens/elapsed:5.1f} 토큰/초)")

print("-" * 60)
print()
print("토큰 수에 거의 비례해 시간이 는다.")
print("한 토큰마다 모델을 한 번씩 부르기 때문이다 (22장 6절).")
print()
print("이 때문에 실무에서는 KV Cache 같은 최적화를 쓴다.")
print("  이미 계산한 K·V를 저장해 두고 재사용하는 방법이다.")
print("  transformers 는 기본으로 이 최적화를 적용한다.")

---

## 9. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| 20.2 | BPE — `newest` → `new`+`est` | 실제 토크나이저에서 확인 ✓ |
| 20.2 | 하위단어로 OOV 해결 | 긴 단어 분해 확인 ✓ |
| 18.3 | Attention 가중치 합 = 1 | 검증 ✓ |
| 18.6 | 마스킹 — 미래 위치 0 | 검증 ✓ |
| 22.5 | 메모리 계산 | 실측과 일치 ✓ |

### 우리가 만든 것 ↔ GPT-2

| 21~22장에서 만든 것 | GPT-2 | 확인 |
|---|---|---|
| `norm1` (LayerNorm) | `ln_1` | ✓ |
| `MaskedSelfAttention` | `attn` | ✓ |
| `norm2` | `ln_2` | ✓ |
| `FeedForward` | `mlp` | ✓ |

**구조가 정확히 같다.** 이름과 규모만 다를 뿐이다.

차이점 하나: 실제 구현은 `W_q, W_k, W_v`를 **하나로 합쳐**(`c_attn`) 한 번에 계산한다.
계산이 더 빠르기 때문이다.

### 기억할 것

| 항목 | 요점 |
|---|---|
| `AutoModelFor___` | 뒤에 붙는 것이 용도 |
| 모델 캐시 | `~/.cache/huggingface` — 한 번만 받음 |
| 공백 | GPT-2는 단어 앞 공백을 토큰에 포함 |
| 한국어 | 영어보다 훨씬 많은 토큰 → 비용·속도 영향 |
| 파라미터 분포 | **FFN이 Attention보다 많다** |
| `output_attentions=True` | Attention 가중치 꺼내기 |
| 생성 속도 | 토큰 수에 비례 (한 토큰 = 모델 1회) |

### 다음 장

**24. 생성 파라미터와 대화 형식** — 이론편 20.5~20.6절.
온도·Top-k·Top-p가 생성 결과를 어떻게 바꾸는지 실험하고,
**ChatML 대화 형식**을 직접 다뤄 본다.

### 지금까지의 수치를 그림으로

**언어별 토큰 효율**과 **파라미터 분포**를 그래프로 보면 차이가 훨씬 분명해진다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- 왼쪽: 언어별 토큰 효율 ---
ax = axes[0]
samples = {
    "영어":   "Artificial intelligence is transforming the world.",
    "한국어": "인공지능이 세상을 바꾸고 있다.",
    "숫자":   "1234567890",
    "코드":   "def hello(): return 'world'",
}
labels, ratios = [], []
for name, text in samples.items():
    ids = tokenizer.encode(text)
    labels.append(name)
    ratios.append(len(ids) / len(text))

colors = ["#0D9488" if r < 0.5 else "#DC2626" for r in ratios]
bars = ax.bar(labels, ratios, color=colors)
for b, r in zip(bars, ratios):
    ax.text(b.get_x() + b.get_width()/2, r + 0.04, f"{r:.2f}",
            ha="center", fontsize=10)
ax.set_ylabel("글자당 토큰 수")
ax.set_title("언어별 토큰 효율 (낮을수록 효율적)")
ax.grid(axis="y", alpha=0.3)

# --- 오른쪽: 파라미터 분포 ---
ax = axes[1]
total = sum(p.numel() for p in model.parameters())
groups = {}
for name, p in model.named_parameters():
    if "wte" in name or "embed_tokens" in name:
        key = "토큰 임베딩"
    elif "wpe" in name or "embed_positions" in name:
        key = "위치 임베딩"
    elif "attn" in name:
        key = "Attention"
    elif "mlp" in name:
        key = "Feed-Forward"
    else:
        key = "기타"
    groups[key] = groups.get(key, 0) + p.numel()

ordered = dict(sorted(groups.items(), key=lambda x: -x[1]))
ax.barh(list(ordered.keys()),
        [v / 1e6 for v in ordered.values()],
        color=["#1E40AF", "#EA580C", "#0D9488", "#94A3B8", "#64748B"])
for i, (k, v) in enumerate(ordered.items()):
    ax.text(v/1e6 + 0.8, i, f"{v/total*100:.1f}%", va="center", fontsize=9)
ax.set_xlabel("파라미터 (백만 개)")
ax.set_title("GPT-2 파라미터 분포")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

kr, en = ratios[labels.index("한국어")], ratios[labels.index("영어")]
print(f"왼쪽: 한국어가 영어보다 글자당 {kr/en:.1f}배 많은 토큰을 쓴다")
print("  같은 내용도 문맥 창을 더 차지하고 비용도 더 든다 (25장 6절)")
print()
print("오른쪽: Attention 보다 Feed-Forward 의 파라미터가 더 많다")
print("  Attention 이 주목받지만 무게 중심은 FFN 에 있다.")